# Comparação dos modelos:

### Importação das bibliotecas:

In [1]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Sistema de arquivos e manipulação de caminhos
import os
import glob
import joblib

# Machine Learning e Métricas
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score, roc_curve, auc
from sklearn.preprocessing import StandardScaler

# Visualização de dados
import matplotlib.pyplot as plt
import seaborn as sns

# Ignorar avisos para manter o notebook limpo
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização para gráficos mais bonitos
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = [14, 10]
plt.rcParams['font.size'] = 12

### 2. Carregamento Centralizado de Dados e Modelos

 Nesta etapa, vamos carregar todos os artefatos necessários para a nossa análise comparativa.

 Fluxo:
 1. Definir os caminhos para os dados e modelos.
 2. Carregar os artefatos de pré-processamento (LabelEncoder, Features).
 3. Carregar e limpar o dataset completo.
 4. **Metodologia Correta de Preparação dos Dados de Teste**:

    a. Dividir os dados brutos em treino e teste.


    b. Usar o conjunto de treino bruto para treinar o `StandardScaler`.

    c. Usar o scaler treinado para transformar os conjuntos de teste.
       Isso garante **zero vazamento de dados**.
 5. Carregar nossos quatro modelos em um dicionário.

In [2]:
# --- 1. Define os caminhos ---
try:
    project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
except:
    project_root = os.getcwd()

model_dir = os.path.join(project_root, 'models')
data_dir = os.path.join(project_root, 'MachineLearningCSV', 'MachineLearningCVE')

print(f"Diretório de modelos: {model_dir}")
print(f"Diretório de dados: {data_dir}")

# --- 2. Carrega os artefatos de pré-processamento ---
print("\nCarregando artefatos de pré-processamento...")
try:
    label_encoder = joblib.load(os.path.join(model_dir, 'label_encoder.joblib'))
    features_realistas = joblib.load(os.path.join(model_dir, 'features_realistas.joblib'))
    print("  - Artefatos carregados com sucesso.")
except Exception as e:
    raise RuntimeError(f"ERRO: Não foi possível carregar os artefatos. Verifique o notebook 01. Erro: {e}")

# --- 3. Carrega e limpa o dataset completo ---
print("\nCarregando e limpando o dataset completo...")
todos_os_arquivos = glob.glob(os.path.join(data_dir, "*.csv"))
dados = pd.concat((pd.read_csv(f, low_memory=False) for f in todos_os_arquivos), ignore_index=True)
dados.columns = dados.columns.str.strip()
for col in dados.columns:
    if dados[col].dtype == 'object' and col != 'Label':
        dados[col] = pd.to_numeric(dados[col], errors='coerce')
dados.replace([np.inf, -np.inf], np.nan, inplace=True)
dados.dropna(inplace=True)
print("  - Dataset carregado e limpo.")

# --- 4. Separa features e labels ---
X_bruto = dados.drop('Label', axis=1)
y_bruto = dados['Label']
y_codificado = label_encoder.transform(y_bruto)
X_realista = X_bruto[features_realistas]

# --- 5. Metodologia de Preparação dos Dados de Teste ---
print("\nPreparando conjuntos de teste (com prevenção de data leakage)...")

#  Dividir os dados brutos em treino e teste
X_treino_realista, X_teste_realista, y_treino, y_teste = train_test_split(
    X_realista, y_codificado, test_size=0.3, random_state=42, stratify=y_codificado
)
print("  - Dados divididos em treino e teste.")

# Treinar o Scaler APENAS com os dados de treino
scaler = StandardScaler().fit(X_treino_realista)
print("  - StandardScaler treinado apenas com dados de treino.")

# Transformar o conjunto de teste para criar a versão padronizada
X_teste_padronizado = scaler.transform(X_teste_realista)
print("  - Conjunto de teste padronizado criado sem vazamento de dados.")

print("\nConjuntos de teste prontos:")
print(f"  - Shape do X_teste_realista (para modelos de árvore): {X_teste_realista.shape}")
print(f"  - Shape do X_teste_padronizado (para MLP): {X_teste_padronizado.shape}")
print(f"  - Shape do y_teste (nosso alvo): {y_teste.shape}")

# --- 6. Carrega os modelos campeões ---
print("\nCarregando modelos salvos...")

modelos = {}
nomes_modelos = [
    'random_forest_model',
    'xgboost_model',
    'catboost_model',
    'mlp_model'
]

for nome in nomes_modelos:
    caminho_modelo = os.path.join(model_dir, f"{nome}.joblib")
    try:
        modelos[nome] = joblib.load(caminho_modelo)
        print(f"  - Modelo '{nome}' carregado.")
    except FileNotFoundError:
        print(f"  - AVISO: Modelo '{nome}' não encontrado em '{caminho_modelo}'.")

print("\n--- Setup concluído ---")

Diretório de modelos: /home/henrique/PycharmProjects/ndr-tcc/models
Diretório de dados: /home/henrique/PycharmProjects/ndr-tcc/MachineLearningCSV/MachineLearningCVE

Carregando artefatos de pré-processamento...
  - Artefatos carregados com sucesso.

Carregando e limpando o dataset completo...
  - Dataset carregado e limpo.

Preparando conjuntos de teste (com prevenção de data leakage)...
  - Dados divididos em treino e teste.
  - StandardScaler treinado apenas com dados de treino.
  - Conjunto de teste padronizado criado sem vazamento de dados.

Conjuntos de teste prontos:
  - Shape do X_teste_realista (para modelos de árvore): (848363, 14)
  - Shape do X_teste_padronizado (para MLP): (848363, 14)
  - Shape do y_teste (nosso alvo): (848363,)

Carregando modelos salvos...
  - Modelo 'random_forest_model' carregado.
  - Modelo 'xgboost_model' carregado.
  - Modelo 'catboost_model' carregado.
  - Modelo 'mlp_model' carregado.

--- Setup concluído ---


### 3. Geração de Previsões e Tabela Comparativa

 Com os dados e modelos devidamente carregados, vamos executar cada modelo em seu respectivo conjunto de teste (padronizado ou não) e coletar as métricas de performance.

 Os resultados serão compilados em um DataFrame do Pandas para criar um ranking claro e objetivo dos modelos.

In [3]:
# Lista para armazenar os dicionários de resultados
lista_de_resultados = []

# Itera sobre cada modelo carregado
for nome, modelo in modelos.items():
    print(f"Avaliando o modelo: {nome}...")

    # Seleciona o conjunto de teste correto
    if nome == 'mlp_model':
        X_teste = X_teste_padronizado
    else:
        X_teste = X_teste_realista

    # Gera previsões
    y_pred = modelo.predict(X_teste)

    # Calcula métricas
    acc = accuracy_score(y_teste, y_pred)
    f1 = f1_score(y_teste, y_pred, average='weighted')

    # Armazena os resultados em um dicionário
    resultado_modelo = {
        'Modelo': nome.replace('_model', '').replace('_', ' ').title(),
        'Acurácia': acc,
        'F1-Score Ponderado': f1
    }
    lista_de_resultados.append(resultado_modelo)

    print(f"  -> Acurácia: {acc:.4f}, F1-Score Ponderado: {f1:.4f}")

# Converte a lista de resultados em um DataFrame
df_resultados = pd.DataFrame(lista_de_resultados)

# Ordena o DataFrame pelo F1-Score para criar o ranking
df_resultados = df_resultados.sort_values(by='F1-Score Ponderado', ascending=False).reset_index(drop=True)

print("\n--- Tabela de Classificação Final ---")
# Mostra a tabela de resultados formatada
display(df_resultados.style.format({
    'Acurácia': '{:.4f}',
    'F1-Score Ponderado': '{:.4f}'
}).background_gradient(cmap='viridis', subset=['F1-Score Ponderado']))

Avaliando o modelo: random_forest_model...
  -> Acurácia: 0.9786, F1-Score Ponderado: 0.9781
Avaliando o modelo: xgboost_model...
  -> Acurácia: 0.9776, F1-Score Ponderado: 0.9771
Avaliando o modelo: catboost_model...
  -> Acurácia: 0.9325, F1-Score Ponderado: 0.9470
Avaliando o modelo: mlp_model...
  -> Acurácia: 0.9412, F1-Score Ponderado: 0.9406

--- Tabela de Classificação Final ---


,Modelo,Acurácia,F1-Score Ponderado
0,Random Forest,0.9786,0.9781
1,Xgboost,0.9776,0.9771
2,Catboost,0.9325,0.9470
3,Mlp,0.9412,0.9406
